In [219]:
from typing import List
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

import cvxpy as cp

from Components.RecedingHorizon import RecedingHorizon
from Components.System import SimpleSquareExample
from Components.Model import SquareCoarseModel
from Components.Controllers import LTI_LQRc_RHG

from Components.Solver import fbrs

In [220]:
# Reload scripts when executed
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [221]:
np.set_printoptions(linewidth=np.inf)

In [222]:
parameter_file = "Components/Setup/two_cells_aquifer.json"

In [223]:
# Initialize Model
model = SquareCoarseModel(parameter_file)

In [224]:
model.A.toarray(), model.B_v[0].toarray()

(array([[0.9796, 0.0204],
        [0.0204, 0.9796]]),
 array([[2.e-05],
        [0.e+00]]))

In [225]:
# Initialize Controller
controller = LTI_LQRc_RHG(
    parameter_file,
    model.A.toarray(), [B.toarray() for B in model.B_v], model.C,
    model.H_x, model.h_x, model.H_u_v, model.h_u_v,
    Q_v = None, R_v = None, M_v = [G.toarray() for G in model.Gamma_v], L_v = [Xi for Xi in model.Xi_v],
    solver='fbrs'
)

In [226]:
controller.A_tilde.toarray(), controller.B_v_tilde[0].toarray()*1e5, controller.B_v_tilde[1].toarray()*1e5

(array([[1.        , 0.        ],
        [0.        , 1.        ],
        [0.9796    , 0.0204    ],
        [0.0204    , 0.9796    ],
        [0.96003232, 0.03996768],
        [0.03996768, 0.96003232],
        [0.941263  , 0.058737  ],
        [0.058737  , 0.941263  ]]),
 array([[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [2.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [1.9592    , 2.        , 0.        ],
        [0.0408    , 0.        , 0.        ],
        [1.92006464, 1.9592    , 2.        ],
        [0.07993536, 0.0408    , 0.        ]]),
 array([[0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        ],
        [2.        , 0.        , 0.        ],
        [0.0408    , 0.        , 0.        ],
        [1.9592    , 2.        , 0.        ],
        [0.07993536, 0.0408    , 0.        ],
        [1.92006464, 1.9592    , 2

In [227]:
controller.F.toarray(), controller.f_x.toarray(), controller.f.toarray()

(array([[0.      , 0.02    , 0.019592, 0.      , 0.      , 0.      ],
        [0.02    , 0.      , 0.02    , 0.      , 0.      , 0.      ],
        [0.019592, 0.02    , 0.      , 0.000408, 0.      , 0.      ],
        [0.      , 0.      , 0.      , 0.      , 0.02    , 0.019592],
        [0.      , 0.      , 0.      , 0.02    , 0.      , 0.02    ],
        [0.000408, 0.      , 0.      , 0.019592, 0.02    , 0.      ]]),
 array([[1000.     ,    0.     ],
        [ 979.6    ,   20.4    ],
        [ 960.03232,   39.96768],
        [   0.     , 1000.     ],
        [  20.4    ,  979.6    ],
        [  39.96768,  960.03232]]),
 array([[-8000.],
        [-8000.],
        [-8000.],
        [-8000.],
        [-8000.],
        [-8000.]]))

In [228]:
(0.5*(controller.F.T + controller.F)).toarray()

array([[0.      , 0.02    , 0.019592, 0.      , 0.      , 0.000204],
       [0.02    , 0.      , 0.02    , 0.      , 0.      , 0.      ],
       [0.019592, 0.02    , 0.      , 0.000204, 0.      , 0.      ],
       [0.      , 0.      , 0.000204, 0.      , 0.02    , 0.019592],
       [0.      , 0.      , 0.      , 0.02    , 0.      , 0.02    ],
       [0.000204, 0.      , 0.      , 0.019592, 0.02    , 0.      ]])

In [229]:
controller.H = sp.csr_array(np.zeros_like(controller.H.toarray()))
controller.h_x_0 = sp.csr_array(np.zeros_like(controller.h_x_0.toarray()))
controller.h = sp.csr_array(np.zeros_like(controller.h.toarray()))

In [230]:
x_0 = 10. * np.ones((2,1))
u_opt, _ = controller.step(x_0)
u_opt

array([[-49484.74439281, -49484.74439281],
       [-50515.25560719, -50515.25560719],
       [-50515.25560719, -50515.25560719]])

In [231]:
(controller.F.T + controller.F).toarray()

array([[0.      , 0.04    , 0.039184, 0.      , 0.      , 0.000408],
       [0.04    , 0.      , 0.04    , 0.      , 0.      , 0.      ],
       [0.039184, 0.04    , 0.      , 0.000408, 0.      , 0.      ],
       [0.      , 0.      , 0.000408, 0.      , 0.04    , 0.039184],
       [0.      , 0.      , 0.      , 0.04    , 0.      , 0.04    ],
       [0.000408, 0.      , 0.      , 0.039184, 0.04    , 0.      ]])

In [232]:
x = cp.Variable(6)
#prob = cp.Problem(0.5*cp.Minimize(cp.quad_form(x, (controller.F.T + controller.F).toarray()) + (controller.f_x @ x_0 + controller.f).T @ x))
#prob.solve()
# => Not DCP

In [233]:
np.linalg.eigvals((0.5*(controller.F.T + controller.F)).toarray())

array([ 0.03986415,  0.03959339, -0.019388  , -0.019796  , -0.02020539, -0.02006815])

In [234]:
F = controller.F.copy()
F[2,3] = 0
F[-1,0] = 0
np.linalg.eigvals(F.toarray())

array([ 0.03972862, -0.019592  , -0.02013662,  0.03972862, -0.019592  , -0.02013662])

In [235]:
test = np.zeros((3,3))
test[0,1] = test[1,0] = test[1,2] = test[2,1] = -5.
test[0,2] = test[2,0] = -4.4
np.linalg.eigvals(test)

array([-9.60540343,  4.4       ,  5.20540343])

In [236]:
mu_h = model.A[0,1]
mu_q = model.B_v[0][0,0]
rho = model.Gamma_v[0][0,0]

(1-mu_h)*mu_q*rho, 0.5*(-mu_q*rho*np.sqrt((1-mu_h)**2 + 8) - (1-mu_h)*mu_q*rho), 0.5*(mu_q*rho*np.sqrt((1-mu_h)**2 + 8) - (1-mu_h)*mu_q*rho)

(np.float64(1.9592e-05),
 np.float64(-3.972861792760533e-05),
 np.float64(2.013661792760533e-05))

In [237]:
np.linalg.eigvals(controller.F.toarray()[:3,:3])*1e-3

array([ 3.97286179e-05, -1.95920000e-05, -2.01366179e-05])

By definition M_{11} and M_{22} are not positive semi-definite (nor negative)

F is not positive semi-definite either and the problem is therefore not convex

In [238]:
import gurobipy as gp

m = gp.Model("matrix1")
u = m.addMVar(shape=6, vtype=gp.GRB.CONTINUOUS, name="U")
m.setObjective(u.T @ (0.25*controller.F.T + 0.25*controller.F).toarray() @ u + -(controller.f_x @ x_0 + controller.f).T @ u, gp.GRB.MINIMIZE)
m.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[x86] - Darwin 24.4.0 24E263)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 6 columns and 0 nonzeros
Model fingerprint: 0x6ca92a15
Model has 8 quadratic objective terms
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [2e+03, 2e+03]
  QObjective range [4e-04, 4e-02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Continuous model is non-convex -- solving as a MIP

Found heuristic solution: objective 0.0000000
Found heuristic solution: objective -1199.998804
Presolve time: 0.00s
Presolved: 17 rows, 16 columns, 33 nonzeros
Presolved model has 8 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
         Consider bounding these variables or reformulating the model.

Variable types: 16 continuous, 

In [239]:
u.X, m.ObjVal

(array([7.53761535e+15, 0.00000000e+00, 0.00000000e+00, 7.53761535e+15, 0.00000000e+00, 0.00000000e+00]),
 -3.015046141143175e+19)

In [240]:
u_test = -1000*np.ones((6,1))
u_test.T @ controller.F @ u_test + (controller.f_x @ x_0 + controller.f).T @ u_test

array([[-11760816.]])

In [241]:
u_opt_f = u_opt.T.reshape((6,1))
u_opt_f.T @ controller.F @ u_opt_f + (controller.f_x @ x_0 + controller.f).T @ u_opt_f

array([[0.]])

In [242]:
(-u.X).T @ controller.F @ (-u.X) + (controller.f_x @ x_0 + controller.f).T @ (-u.X)

array([-3.01504614e+19])

In [243]:
u_analytic = np.linalg.inv((controller.F).toarray()) @ (-(controller.f_x @ x_0 + controller.f))
u_analytic

array([[-49484.74439281],
       [-50515.25560719],
       [-50515.25560719],
       [-49484.74439281],
       [-50515.25560719],
       [-50515.25560719]])

In [244]:
0.5*(controller.F+controller.F.T) @ u_analytic + controller.f_x @ x_0 + controller.f

array([[-10.30511214],
       [  0.        ],
       [ 10.09488786],
       [-10.30511214],
       [  0.        ],
       [ 10.09488786]])

F is not symmetric and cannot be integrated.

# 13.05: Check if optimal control is the Nash equilibrium

In [245]:
parameter_file = "Components/Setup/two_cells_aquifer_smaller.json"

In [246]:
# Initialize Model
model_s = SquareCoarseModel(parameter_file, boundary_condition="Constant Heads")

In [247]:
model.A[0,1], model_s.A[0,1]

(np.float64(0.0204), np.float64(0.5233280000000001))

In [248]:
# Initialize Controller
controller = LTI_LQRc_RHG(
    parameter_file,
    model_s.A.toarray(), [B.toarray() for B in model_s.B_v], model_s.C,
    model_s.H_x, model_s.h_x, model_s.H_u_v, model_s.h_u_v,
    Q_v = None, R_v = None, M_v = [G.toarray() for G in model_s.Gamma_v], L_v = [Xi for Xi in model_s.Xi_v],
    solver='fbrs'
)

In [249]:
x_0 = 10. * np.ones((2,1))
u_opt, _ = controller.step(x_0)
u_opt

array([[-2.63952903e-14, -5.48297784e-14],
       [-1.00000000e+03, -1.00000000e+03],
       [-1.59168081e+03, -1.59168081e+03]])

In [250]:
u0 = np.array([[-1000., 0., 0., -1000., 0., 0.]]).T

In [251]:
# Form gradient and constraint based on initial state
f = sp.csr_array(controller.f_x @ x_0 + controller.f)
h = sp.csr_array(controller.h_x_0 @ x_0 + controller.h)
# Solve optimization problem
U_opt, U_dual, _, out = fbrs(controller.F.toarray(), f, controller.H.toarray(), h.toarray(), u0, controller.v0)
out['eflag'], U_opt

(0,
 array([[-6.67299264e-13],
        [-1.00000000e+03],
        [-1.59168081e+03],
        [-6.27649198e-13],
        [-1.00000000e+03],
        [-1.59168081e+03]]))

In [252]:
u_analytic = np.linalg.inv((controller.F).toarray()) @ (-(controller.f_x @ x_0 + controller.f))
u_analytic

array([[ -293.07729179],
       [ -706.92270821],
       [-1125.1953123 ],
       [ -293.07729179],
       [ -706.92270821],
       [-1125.1953123 ]])

In [253]:
x_ana = controller.A_tilde @ x_0 + sp.hstack(controller.B_v_tilde) @ U_opt + controller.C_tilde
x_ana = x_ana.reshape(controller.K+1, controller.nx).T
x_ana

array([[10.        , 10.        ,  8.        ,  4.81663837],
       [10.        , 10.        ,  8.        ,  4.81663837]])

In [254]:
h = controller.h_x_0 @ x_0 + controller.h

In [255]:
(controller.H @ u_analytic <= h).T

array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True]])

In [256]:
u = u_analytic.reshape(2, controller.K).T

In [257]:
with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()
    m = gp.Model("matrix1", env)
U_v_splitted = np.split(u, np.cumsum(controller.nu_v)[:-1], axis=1)
H_v = np.split(controller.H.toarray(), np.cumsum(controller.nu_v)[:-1]*controller.K, axis=1)
for v, nu_v in enumerate(controller.nu_v):
    # Define variable and parameters
    u_v = m.addMVar(shape=controller.K*nu_v, lb=-np.inf,
                            vtype=gp.GRB.CONTINUOUS, name="u_v")
    u_minus_v = np.concatenate([U_block.reshape(-1, 1)
                                for p, U_block in enumerate(U_v_splitted)
                                if p != v])
    # Set individual cost
    cost = u_v.T @ controller.R_vv[v] @ u_v + u_v.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v + controller.L_v_tilde[v] @ u_v
    m.setObjective(cost, gp.GRB.MINIMIZE)
    # Set constraints
    h_v = h - np.hstack([H_v[p] for p in range(controller.Nv) if p!=v]) @ u_minus_v
    cstrt = (H_v[v] @ u_v)[:,np.newaxis] <= h_v
    m.addConstr(cstrt, name="H")
    # Compare results
    m.optimize()
    u_v_r = U_v_splitted[v].reshape(-1, 1)
    real_cost = u_v_r.T @ controller.R_vv[v] @ u_v_r + u_v_r.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v_r + controller.L_v_tilde[v] @ u_v_r
    print(real_cost)
    print(m.ObjVal)
    print(u_v.X)

[[-1585.63401435]]
-20000.0
[-10000.      0.      0.]
[[-1585.63401435]]
-20000.0
[-10000.      0.      0.]


In [258]:
real_cost - (u_v.X.reshape(-1,1).T @ controller.R_vv[v] @ u_v.X.reshape(-1,1) + u_v.X.reshape(-1,1).T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v.X.reshape(-1,1) + controller.L_v_tilde[v] @ u_v.X.reshape(-1,1))

array([[18414.36598565]])

In unconstrained setup $Fu+f = 0$ is not the Nash equilibrium???

In [259]:
(controller.R_vv[v].T + controller.R_vv[v]) @ u_v_r + controller.R_v_minus_v[v] @ u_minus_v + controller.M_x_v[v].T @ x_0 + controller.L_v_tilde[v].T

array([[ 0.0000000e+00],
       [ 0.0000000e+00],
       [-8.8817842e-16]])

In [260]:
(controller.R_vv[v].T + controller.R_vv[v]) @ u_v.X.reshape(-1,1) + controller.R_v_minus_v[v] @ u_minus_v + controller.M_x_v[v].T @ x_0 + controller.L_v_tilde[v].T

array([[  2.        ],
       [-10.78322653],
       [ -8.70247036]])

Is the zero-crossing $u$ a maximum?

In [261]:
np.linalg.eigvals(controller.R_vv[v].T + controller.R_vv[v])

array([ 0.00233841, -0.00098803, -0.00135038])

No

It might be only a local minima

In [262]:
deltas = [0.1*np.array([[1, 0, 0]]).T, 0.1*np.array([[0, 1, 0]]).T, 0.1*np.array([[0, 0, 1]]).T,
          0.1*np.array([[1, 1, 0]]).T, 0.1*np.array([[1, 0, 1]]).T, 0.1*np.array([[0, 1, 1]]).T,
          0.1*np.array([[1, -1, 0]]).T, 0.1*np.array([[1, 0, -1]]).T, 0.1*np.array([[0, 1, -1]]).T,]

cost = u_v_r.T @ controller.R_vv[v] @ u_v_r + u_v_r.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v_r + controller.L_v_tilde[v] @ u_v_r
for delta in deltas:
    u_v_d = u_v_r + delta
    uv_minus_d = u_v_r - delta
    print(cost - (u_v_d.T @ controller.R_vv[v] @ u_v_d + u_v_d.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v_d + controller.L_v_tilde[v] @ u_v_d))
    print(cost - (uv_minus_d.T @ controller.R_vv[v] @ uv_minus_d + uv_minus_d.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ uv_minus_d + controller.L_v_tilde[v] @ uv_minus_d))

[[3.63797881e-12]]
[[0.]]
[[3.63797881e-12]]
[[0.]]
[[3.63797881e-12]]
[[3.63797881e-12]]
[[-1.25653314e-05]]
[[-1.25653314e-05]]
[[-9.88031024e-06]]
[[-9.88031024e-06]]
[[-1.25653314e-05]]
[[-1.25653314e-05]]
[[1.25653351e-05]]
[[1.25653351e-05]]
[[9.88031388e-06]]
[[9.88031024e-06]]
[[1.25653387e-05]]
[[1.25653351e-05]]


It seems that it is a saddle point.

In [263]:
with gp.Env(empty=True) as env:
    #env.setParam('OutputFlag', 0)
    env.start()
    m = gp.Model("matrix1", env)
U_v_splitted = np.split(u, np.cumsum(controller.nu_v)[:-1], axis=1)
H_v = np.split(controller.H.toarray(), np.cumsum(controller.nu_v)[:-1]*controller.K, axis=1)
for v, nu_v in enumerate(controller.nu_v):
    # Define variable and parameters
    u_v = m.addMVar(shape=controller.K*nu_v, lb=-np.inf,
                            vtype=gp.GRB.CONTINUOUS, name="u_v")
    u_minus_v = np.concatenate([U_block.reshape(-1, 1)
                                for p, U_block in enumerate(U_v_splitted)
                                if p != v])
    # Set individual cost
    cost = u_v.T @ controller.R_vv[v] @ u_v + u_v.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v + controller.L_v_tilde[v] @ u_v
    m.setObjective(cost, gp.GRB.MINIMIZE)
    # Set constraints
    h_v = h - np.hstack([H_v[p] for p in range(controller.Nv) if p!=v]) @ u_minus_v
    cstrt = (H_v[v] @ u_v)[:,np.newaxis] <= h_v
    #m.addConstr(cstrt, name="H")
    # Compare results
    m.optimize()
    u_v_r = U_v_splitted[v].reshape(-1, 1)
    real_cost = u_v_r.T @ controller.R_vv[v] @ u_v_r + u_v_r.T @ controller.R_v_minus_v[v] @ u_minus_v + x_0.T @ controller.M_x_v[v] @ u_v_r + controller.L_v_tilde[v] @ u_v_r
    print(real_cost)
    print(m.ObjVal)
    print(u_v.X)

Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-28
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[x86] - Darwin 24.4.0 24E263)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 0 rows, 3 columns and 0 nonzeros
Model fingerprint: 0xaa2d2963
Model has 3 quadratic objective terms
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [1e+00, 2e+00]
  QObjective range [2e-03, 3e-03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Continuous model is non-convex -- solving as a MIP

Found heuristic solution: objective 0.0000000
Presolve time: 0.00s
Presolved: 7 rows, 8 columns, 16 nonzeros
Presolved model has 3 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
         Consider bounding these variables or reformulating the

Unbounded problem ??

Maybe because the first pumping is "free": it only produces benefits. So the optimal strategy is to pump infinitely at the first time-step and then stop.

This problem comes from the modelling: the water pumped in one time-step is modelled as being pumped all together instantaneously.

In reality, the pumping cost is bounded by:

$cost(h=h_0) \le cost_{real} \le cost(h=h_1)$

Instead of taking the lower bound, let's optimize the upper bound.

This is equivalent to defining the cost function as:

$l_k(h_{k,t+1},q_{k,t}) = \rho_{c,k} (h_{0,k} - h_{k,t+1}) q_{k,t} - \rho_{p,k} q_{k,t}$

Which is a bit odd because we now have an inter-time-step cross-term, but it might disappear when we use the aggregated U.

It is also equivalent to have as a state $x_t^v = [h_t^T \ {q_{t-1}^v}^T]^T$, with $x_0 = [h_0^T \ 0]^T$ and:

$x_{t+1}^v = \begin{bmatrix} A & 0 \\ 0 & 0 \end{bmatrix} x_t^v + \begin{bmatrix} B \\ I \end{bmatrix} u_t^v + \begin{bmatrix} C \\ 0 \end{bmatrix}$

And the cost function becomes:

$J(x_t, u_t) = h_t^T\Gamma q_{t-1} + \Xi q_t = x_t \begin{bmatrix} 0 & \Gamma \\ 0 & 0 \end{bmatrix} x_t + \Xi u_t$

In [264]:
model_s.Gamma_v[0].toarray(), model_s.Gamma_v[1].toarray()

(array([[1.],
        [0.]]),
 array([[0.],
        [1.]]))

In [ ]:
nx = model_s.n_col * model_s.n_row
# Form new dynamics matrices
A_tilde = sp.block_array([[model_s.A, sp.csr_array((nx, model_s.nu_v[0]))], [sp.csr_array((model_s.nu_v[0], nx)), sp.csr_array((model_s.nu_v[0], model_s.nu_v[0]))]]) #TODO: here it works because nu_1=nu_2=1, but it would require an other class to handle x_v and A_v
B_tilde = [sp.block_array([[model_s.B_v[v]], [sp.eye((model_s.nu_v[v]))]]) for v in range(model_s.n_agent)]
C_tilde = sp.block_array([[model_s.C], [sp.csr_array((model_s.nu_v[0], 1))]])

In [272]:
# Form new constraint matrices
H_x_tilde = np.block([[model_s.H_x.toarray(), np.zeros((nx, model_s.nu_v[0]))]])

In [278]:
# Form new Cost matrices
Gamma_v_tilde = [sp.csr_array((nx+model_s.nu_v[v], model_s.nu_v[v])) for v in range(model_s.n_agent)]
Q_v = [sp.block_array([[sp.csr_array((nx, nx)), model_s.Gamma_v[v]], [sp.csr_array((model_s.nu_v[v], nx)), sp.csr_array((model_s.nu_v[v], model_s.nu_v[v]))]]).toarray() for v in range(model_s.n_agent)]

In [279]:
# Initialize Controller
controller_tilde = LTI_LQRc_RHG(
    parameter_file,
    A_tilde.toarray(), [B.toarray() for B in B_tilde], C_tilde.toarray(),
    H_x_tilde, model_s.h_x, model_s.H_u_v, model_s.h_u_v,
    Q_v = Q_v, R_v = None, M_v = [G.toarray() for G in Gamma_v_tilde], L_v = [Xi for Xi in model_s.Xi_v],
    solver='fbrs'
)

In [281]:
controller.F.toarray(), controller_tilde.F.toarray()

(array([[0.        , 1.25653333, 0.98803118, 0.        , 0.        , 0.        ],
        [1.25653333, 0.        , 1.25653333, 0.74346667, 0.        , 0.        ],
        [0.98803118, 1.25653333, 0.        , 1.01196882, 0.74346667, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 1.25653333, 0.98803118],
        [0.74346667, 0.        , 0.        , 1.25653333, 0.        , 1.25653333],
        [1.01196882, 0.74346667, 0.        , 0.98803118, 1.25653333, 0.        ]]),
 array([[2.51306667, 0.98803118, 1.00055842, 2.51306667, 1.97606236, 2.00111683],
        [0.98803118, 2.51306667, 0.98803118, 0.        , 2.51306667, 1.97606236],
        [1.00055842, 0.98803118, 2.51306667, 0.        , 0.        , 2.51306667],
        [2.51306667, 1.97606236, 2.00111683, 2.51306667, 0.98803118, 1.00055842],
        [0.        , 2.51306667, 1.97606236, 0.98803118, 2.51306667, 0.98803118],
        [0.        , 0.        , 2.51306667, 1.00055842, 0.98803118, 2.51306667]]))

In [283]:
np.linalg.eigvals((controller_tilde.F.T + controller_tilde.F).toarray())

array([ 1.79899691e+01, -2.12720466e-16,  6.05003300e+00,  6.11679793e+00,  4.52786597e-16, -2.02282501e-16])

In [287]:
x_tilde_0 = np.block([[x_0], [np.zeros((1, 1))]])

In [314]:
u_analytic = np.linalg.inv((controller_tilde.F).toarray()) @ (-(controller_tilde.f_x @ x_tilde_0 + controller_tilde.f))
u_analytic

array([[10112.],
       [ 7744.],
       [ 1920.],
       [ 8192.],
       [-9600.],
       [    0.]])

The analytical optimal solution require positive pumping (i.e. injecting water) ???

In [312]:
np.linalg.inv((controller_tilde.F).toarray()) @ controller_tilde.F

array([[ 1.  ,  0.  , -2.  , -1.5 , -0.5 , -1.  ],
       [-1.  , -1.  , -1.  ,  0.5 , -0.25, -2.  ],
       [ 1.  ,  1.  , -1.  , -1.  , -0.75,  1.  ],
       [ 0.  , -1.  ,  0.  ,  1.75,  0.  ,  0.  ],
       [ 1.  ,  2.  ,  1.  , -0.5 ,  0.5 ,  1.  ],
       [-1.  , -1.  ,  0.  ,  0.75,  1.  ,  2.  ]])

F is not invertible

In [315]:
u_analytic = np.linalg.pinv((controller_tilde.F).toarray()) @ (-(controller_tilde.f_x @ x_tilde_0 + controller_tilde.f))
u_analytic

array([[ -522.56446344],
       [-1061.15639196],
       [-2074.89285162],
       [ -522.56446344],
       [-1061.15639196],
       [-2074.89285162]])

In [316]:
controller_tilde.F @ u_analytic + (controller_tilde.f_x @ x_tilde_0 + controller_tilde.f)

array([[ 0.00000000e+00],
       [ 0.00000000e+00],
       [ 0.00000000e+00],
       [ 0.00000000e+00],
       [-3.63797881e-12],
       [-3.63797881e-12]])

In [318]:
u = (u_analytic.reshape(2, controller.K).T)

In [320]:
with gp.Env(empty=True) as env:
    env.setParam('OutputFlag', 0)
    env.start()
    m = gp.Model("matrix1", env)
U_v_splitted = np.split(u, np.cumsum(controller_tilde.nu_v)[:-1], axis=1)
H_v = np.split(controller_tilde.H.toarray(), np.cumsum(controller_tilde.nu_v)[:-1]*controller_tilde.K, axis=1)
for v, nu_v in enumerate(controller_tilde.nu_v):
    # Define variable and parameters
    u_v = m.addMVar(shape=controller_tilde.K*nu_v, lb=-np.inf,
                            vtype=gp.GRB.CONTINUOUS, name="u_v")
    u_minus_v = np.concatenate([U_block.reshape(-1, 1)
                                for p, U_block in enumerate(U_v_splitted)
                                if p != v])
    # Set individual cost
    cost = u_v.T @ controller_tilde.R_vv[v] @ u_v + u_v.T @ controller_tilde.R_v_minus_v[v] @ u_minus_v + x_tilde_0.T @ controller_tilde.M_x_v[v] @ u_v + controller_tilde.L_v_tilde[v] @ u_v
    m.setObjective(cost, gp.GRB.MINIMIZE)
    # Set constraints
    h_v = h - np.hstack([H_v[p] for p in range(controller_tilde.Nv) if p!=v]) @ u_minus_v
    cstrt = (H_v[v] @ u_v)[:,np.newaxis] <= h_v
    m.addConstr(cstrt, name="H")
    # Compare results
    m.optimize()
    u_v_r = U_v_splitted[v].reshape(-1, 1)
    real_cost = u_v_r.T @ controller_tilde.R_vv[v] @ u_v_r + u_v_r.T @ controller_tilde.R_v_minus_v[v] @ u_minus_v + x_tilde_0.T @ controller_tilde.M_x_v[v] @ u_v_r + controller_tilde.L_v_tilde[v] @ u_v_r
    print(real_cost)
    print(m.ObjVal)
    print(u_v.X)

[[-10975.84112105]]
-10975.841121048354
[ -522.56446345 -1061.15639196 -2074.89285161]
[[-10975.84112105]]
-10975.841121048055
[ -522.56446345 -1061.15639196 -2074.89285161]


Now analytical solution is indeed a Nash equilibrium!